In [ ]:
!pip install --upgrade pandas numpy scikit-learn

In [13]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

df = pd.read_csv('/content/sample_data/Dataset.csv')

target = 'Aggregate rating'
features = ['Country Code', 'City', 'Cuisines', 'Average Cost for two',
            'Has Table booking', 'Has Online delivery', 'Price range', 'Votes']

df = df.dropna(subset=[target])
X = df[features]
y = df[target]

categorical_cols = ['City', 'Cuisines', 'Has Table booking', 'Has Online delivery']
numerical_cols = ['Country Code', 'Average Cost for two', 'Price range', 'Votes']

numerical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_cols),
        ('cat', categorical_transformer, categorical_cols)
    ])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

models = {
    "Linear Regression": LinearRegression(),
    "Random Forest Regressor": RandomForestRegressor(n_estimators=100, random_state=42)
}

print("\nTraining models... (This might take a minute due to the size of Cuisines categories)")
for name, model in models.items():
    pipeline = Pipeline(steps=[('preprocessor', preprocessor), ('model', model)])
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)

    print(f"\n=== {name} Performance ===")
    print(f"Mean Squared Error (MSE): {mean_squared_error(y_test, y_pred):.4f}")
    print(f"Root Mean Squared Error (RMSE): {np.sqrt(mean_squared_error(y_test, y_pred)):.4f}")
    print(f"R-squared Score (R2): {r2_score(y_test, y_pred):.4f}")

print("\nCalculating top driving features from Random Forest...")
rf_pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                              ('model', RandomForestRegressor(n_estimators=100, random_state=42))])
rf_pipeline.fit(X_train, y_train)

cat_encoder = rf_pipeline.named_steps['preprocessor'].named_transformers_['cat'].named_steps['onehot']
encoded_cat_cols = list(cat_encoder.get_feature_names_out(categorical_cols))
all_features = numerical_cols + encoded_cat_cols

importances = rf_pipeline.named_steps['model'].feature_importances_
feature_imp_df = pd.DataFrame({'Feature': all_features, 'Importance': importances}).sort_values(by='Importance', ascending=False)

print("\nTop 10 Most Influential Features on Restaurant Ratings:")
print(feature_imp_df.head(10).to_string(index=False))


Training models... (This might take a minute due to the size of Cuisines categories)

=== Linear Regression Performance ===
Mean Squared Error (MSE): 1.4860
Root Mean Squared Error (RMSE): 1.2190
R-squared Score (R2): 0.3471

=== Random Forest Regressor Performance ===
Mean Squared Error (MSE): 0.1014
Root Mean Squared Error (RMSE): 0.3185
R-squared Score (R2): 0.9554

Calculating top driving features from Random Forest...

Top 10 Most Influential Features on Restaurant Ratings:
                                Feature  Importance
                                  Votes    0.949407
                           Country Code    0.008543
                   Average Cost for two    0.006877
                         City_New Delhi    0.001342
                             City_Noida    0.001159
                            Price range    0.001135
         Cuisines_North Indian, Chinese    0.000951
                Has Online delivery_Yes    0.000877
                           City_Gurgaon    0.00